In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [71]:
arkansas_files=["arkansas_batch1.csv","arkansas_batch2.csv","arkansas_other_fixed.csv"]
arkansas_sampling_plan={0: 616, 5: 4677, 3: 2423, 1: 1522, 2: 762}

cal_files = [
    "california_alfafa.csv", "california_almonds.csv", "california_grapes.csv",
    "california_pistachios.csv", "california_rice.csv", "california_other_fixed.csv"
]
california_sampling_plan={0:3512, 69:2054, 3:2037, 36:974, 75:783, 204:640}

## merge batches

### califorina

In [72]:

data_frames = [pd.read_csv(f) for f in cal_files]

# 2. Combine all rows into one giant dataframe immediately
# This puts all Alfalfa rows first, then Almonds, etc.
merged_all = pd.concat(data_frames, axis=0, ignore_index=True)

# 3. GLOBAL SORT
# Sorting by 'time' creates the 36 blocks.
# Sorting by '.geo' within each time block ensures the physical points 
# stay in the exact same order across all 36 periods.
merged_stacked = merged_all.sort_values(['time', '.geo']).reset_index(drop=True)

# 4. Verification
print(merged_stacked.info())
print("Shape of merged data:", merged_stacked.shape)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967452 entries, 0 to 3967451
Data columns (total 16 columns):
 #   Column        Dtype  
---  ------        -----  
 0   system:index  object 
 1   B11           float64
 2   B12           float64
 3   B2            float64
 4   B3            float64
 5   B4            float64
 6   B5            float64
 7   B6            float64
 8   B7            float64
 9   B8            float64
 10  B8A           float64
 11  SCL           float64
 12  crop          int64  
 13  cropland      int64  
 14  time          object 
 15  .geo          object 
dtypes: float64(11), int64(2), object(3)
memory usage: 484.3+ MB
None
Shape of merged data: (3967452, 16)


In [73]:
batch_cal = merged_stacked.to_numpy()

print(merged_stacked.info())
print("Shape of merged data:", batch_cal.shape)

batch1_cal=merged_stacked.to_numpy()[:int(merged_stacked.shape[0]/36)]
np.unique(batch1_cal[:,12], return_counts=True)  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967452 entries, 0 to 3967451
Data columns (total 16 columns):
 #   Column        Dtype  
---  ------        -----  
 0   system:index  object 
 1   B11           float64
 2   B12           float64
 3   B2            float64
 4   B3            float64
 5   B4            float64
 6   B5            float64
 7   B6            float64
 8   B7            float64
 9   B8            float64
 10  B8A           float64
 11  SCL           float64
 12  crop          int64  
 13  cropland      int64  
 14  time          object 
 15  .geo          object 
dtypes: float64(11), int64(2), object(3)
memory usage: 484.3+ MB
None
Shape of merged data: (3967452, 16)


(array([0, 3, 36, 69, 75, 204], dtype=object),
 array([ 4902,  6422, 18217, 26306,  2478, 51882]))

### arkansas

In [ ]:
data1 = pd.read_csv("arkansas_batch1.csv")
data2 = pd.read_csv("arkansas_batch2.csv")
data3 = pd.read_csv("arkansas_other_fixed.csv")
data1_sorted = data1.sort_values(['time', 'system:index']).reset_index(drop=True)
data2_sorted = data2.sort_values(['time', 'system:index']).reset_index(drop=True)
data3_sorted = data3.sort_values(['time', 'system:index']).reset_index(drop=True)


# Get unique 10-day intervals
unique_times = data1_sorted['time'].unique()

# List to collect stacked blocks
stacked_blocks = []

for t in unique_times:
    block1 = data1_sorted[data1_sorted['time'] == t]
    block2 = data2_sorted[data2_sorted['time'] == t]
    block3 = data3_sorted[data3_sorted['time'] == t]
    
    # Keep the order of points as they appear in each block
    stacked_block = pd.concat([block1, block2, block3], axis=0, ignore_index=True)
    stacked_blocks.append(stacked_block)

# Combine all 10-day blocks
merged_stacked = pd.concat(stacked_blocks, axis=0, ignore_index=True)

# Convert to NumPy if needed
batch_ark = merged_stacked.to_numpy()

print(merged_stacked.info())
print("Shape of merged data:", batch_ark.shape)

batch1_ark=merged_stacked.to_numpy()[:int(merged_stacked.shape[0]/36)]
np.unique(batch1_ark[:,12], return_counts=True)  




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033596 entries, 0 to 1033595
Data columns (total 16 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   system:index  1033596 non-null  object 
 1   B11           1033596 non-null  float64
 2   B12           1033596 non-null  float64
 3   B2            1033596 non-null  float64
 4   B3            1033596 non-null  float64
 5   B4            1033596 non-null  float64
 6   B5            1033596 non-null  float64
 7   B6            1033596 non-null  float64
 8   B7            1033596 non-null  float64
 9   B8            1033596 non-null  float64
 10  B8A           1033596 non-null  float64
 11  SCL           1033596 non-null  float64
 12  crop          1033596 non-null  int64  
 13  cropland      1033596 non-null  int64  
 14  time          1033596 non-null  object 
 15  .geo          1033596 non-null  object 
dtypes: float64(11), int64(2), object(3)
memory usage: 126.2+ MB
None
Shape o

(array([0, 1, 2, 3, 5], dtype=object),
 array([ 2717,  1701,  1929,  3679, 18685]))

## remove extra (preference to those with many missing values)

### check for data alignment

In [74]:
# Assuming 36 time steps
n_steps = 36
n_total_rows = merged_stacked.shape[0]
n_points = int(n_total_rows / n_steps)

# Split the .geo column into 36 chunks (one for each time interval)
geo_column = merged_stacked['.geo'].values
chunks = [geo_column[i * n_points : (i + 1) * n_points] for i in range(n_steps)]

# Check if all chunks are identical to the first chunk
is_aligned = all(np.array_equal(chunks[0], chunk) for chunk in chunks)

if is_aligned:
    print(f"✅ Data is PERFECTLY aligned. Each block has {n_points} identical points in order.")
else:
    print("❌ Data is NOT aligned. Points are missing or ordered differently in some time steps.")

✅ Data is PERFECTLY aligned. Each block has 110207 identical points in order.


### reshape

In [75]:
# 1. Drop the non-numeric columns like '.geo' and 'time' to get your 10 bands/features
# Adjust the column selection to keep exactly your 10 bands
features_df = merged_stacked.drop(columns=['.geo', 'time', 'system:index', 'cropland']) 

# 2. Convert to NumPy: Shape (N_total_rows, 10)
raw_array = features_df.to_numpy()

# 3. Reshape: (Time, Points, Features) -> (Points, Time, Features)
# This results in (N_points, 36, 10)
final_data = raw_array.reshape(n_steps, n_points, 12).transpose(1, 0, 2)

print("Final Shape:", final_data.shape) # Should be (N_points, 36, 10)
print("First point's history:\n", final_data[0])

Final Shape: (110207, 36, 12)
First point's history:
 [[2.3000e+03 1.3700e+03 3.6900e+02 8.2900e+02 5.9750e+02 1.3730e+03
  4.1060e+03 4.7040e+03 4.9460e+03 4.7670e+03 4.0000e+00 6.9000e+01]
 [1.8190e+03 1.4325e+03 5.7575e+03 5.6890e+03 5.4210e+03 5.8985e+03
  7.1950e+03 7.3190e+03 7.7800e+03 7.1395e+03 6.0000e+00 6.9000e+01]
 [2.2120e+03 1.1775e+03 2.3250e+02 7.0300e+02 3.3600e+02 1.2150e+03
  4.5250e+03 5.2725e+03 5.5080e+03 5.3485e+03 4.0000e+00 6.9000e+01]
 [2.1590e+03 1.0740e+03 2.6200e+02 6.5700e+02 3.0200e+02 1.0420e+03
  4.3680e+03 5.2790e+03 5.6920e+03 5.3330e+03 4.0000e+00 6.9000e+01]
 [2.1290e+03 1.0330e+03 1.8200e+02 6.1100e+02 2.1900e+02 1.0580e+03
  4.7780e+03 5.7780e+03 6.1280e+03 5.8630e+03 4.0000e+00 6.9000e+01]
 [1.9495e+03 9.1950e+02 2.1600e+02 6.2700e+02 1.9600e+02 1.0325e+03
  4.7955e+03 5.8955e+03 6.2880e+03 5.9575e+03 4.0000e+00 6.9000e+01]
 [2.2300e+03 1.0410e+03 2.6800e+02 6.6400e+02 2.3200e+02 1.0490e+03
  4.9980e+03 6.1960e+03 6.6360e+03 6.2610e+03 4.0000e+00

### interpolate missing values and pick final values

In [ ]:
"""
psudo code:

add a mask that indicates whether the data points has missing values
if target_num_samples>sample:
    sample=+clean data
    if target_num_samples>sample:
        order data with missing value from points with least to most missing values 
        complete samples with it until we reach target_num_samples
        interpolate the missing data
        
        
"""

In [76]:
from scipy.interpolate import interp1d
# ==========================================
# 1. PREPARATION & GAP SCORING
# ==========================================

# Calculate how many 10-day windows are missing (-9999) for each point
# final_data shape is (N, 36, 12)
gap_counts = np.sum(np.any(final_data == -9999, axis=2), axis=1)

# Prepare the Presence Mask (1.0 = Real Data, 0.0 = Gap)
# We use the first band (index 0) to check for gaps
presence_mask = (final_data[:, :, 0] != -9999).astype(np.float32).reshape(-1, 36, 1)

# ==========================================
# 2. INTERPOLATION FUNCTION
# ==========================================

def fill_time_series(data_3d):
    """
    Linearly interpolates missing values (-9999) across the time axis.
    """
    working_data = data_3d.copy().astype(np.float32)
    working_data[working_data == -9999] = np.nan
    
    n_points, n_steps, n_feats = working_data.shape
    
    for p in range(n_points):
        for f in range(n_feats):
            y = working_data[p, :, f]
            nans = np.isnan(y)
            
            # If the entire year is missing for this band, fill with 0
            if np.all(nans):
                y[:] = 0 
            # If there are some gaps, interpolate them
            elif np.any(nans):
                x_known = np.where(~nans)[0]
                # Linear interpolation with extrapolation for start/end of year
                f_interp = interp1d(x_known, y[~nans], kind='linear', fill_value="extrapolate")
                y[nans] = f_interp(np.where(nans)[0])
            
            working_data[p, :, f] = y
    return working_data

# Run interpolation on everything to prepare the features
interpolated_all = fill_time_series(final_data)

# Add the presence mask as the 13th column
# Resulting shape: (N, 36, 13)
data_with_mask = np.concatenate([interpolated_all, presence_mask], axis=2)

# ==========================================
# 3. QUALITY-PRIORITIZED SAMPLING
# ==========================================

sampling_plan = california_sampling_plan
crop_col_idx = 11 # The 'crop' label is at index 11
point_crop_ids = data_with_mask[:, 0, crop_col_idx]

final_samples = []

print("--- Starting Quality-Aware Sampling ---")
for crop_id, target_count in sampling_plan.items():
    # Find all available indices for this specific crop
    crop_indices = np.where(point_crop_ids == crop_id)[0]
    
    # SORT indices by gap count: points with 0 gaps come first, then 1, 2, etc.
    sorted_indices = crop_indices[np.argsort(gap_counts[crop_indices])]
    
    # Select the highest quality points (up to our target count)
    selected = sorted_indices[:target_count]
    
    # Calculate stats for the selection
    perfect_count = np.sum(gap_counts[selected] == 0)
    avg_gaps = np.mean(gap_counts[selected])
    
    print(f"Crop {crop_id}: Selected {len(selected)} points.")
    print(f"   -> {perfect_count} are perfect (0 gaps).")
    print(f"   -> Average gaps in selection: {avg_gaps:.2f}")
    
    final_samples.append(data_with_mask[selected])

# ==========================================
# 4. FINAL ASSEMBLY
# ==========================================

# Combine all classes into one dataset
balanced_dataset = np.concatenate(final_samples, axis=0)

# Shuffle the data so classes aren't grouped together
np.random.shuffle(balanced_dataset)

print("-" * 30)
print(f"Final Dataset Shape: {balanced_dataset.shape}")
print(f"Columns: Bands 1-10, SCL(11), Label(12), PresenceMask(13)")

--- Starting Quality-Aware Sampling ---
Crop 0: Selected 3512 points.
   -> 15 are perfect (0 gaps).
   -> Average gaps in selection: 3.65
Crop 69: Selected 2054 points.
   -> 2054 are perfect (0 gaps).
   -> Average gaps in selection: 0.00
Crop 3: Selected 2037 points.
   -> 60 are perfect (0 gaps).
   -> Average gaps in selection: 1.47
Crop 36: Selected 974 points.
   -> 974 are perfect (0 gaps).
   -> Average gaps in selection: 0.00
Crop 75: Selected 783 points.
   -> 705 are perfect (0 gaps).
   -> Average gaps in selection: 0.10
Crop 204: Selected 640 points.
   -> 640 are perfect (0 gaps).
   -> Average gaps in selection: 0.00
------------------------------
Final Dataset Shape: (10000, 36, 13)
Columns: Bands 1-10, SCL(11), Label(12), PresenceMask(13)


### dataset quality stats

In [77]:
# 1. Extract the Presence Mask from the final dataset 
# (It's the very last column we concatenated)
final_presence_mask = balanced_dataset[:, :, -1] 

# 2. Count zeros (gaps) per point
# Since 1.0 = data and 0.0 = gap, (1 - mask) gives us 1s for gaps
gaps_per_point = np.sum(final_presence_mask == 0, axis=1)

# 3. Calculate statistics
avg_gaps = np.mean(gaps_per_point)
max_gaps = np.max(gaps_per_point)
min_gaps = np.min(gaps_per_point)

print(f"📊 Final Dataset Gap Stats:")
print(f"Average gaps: {avg_gaps:.2f} per point")
print(f"Max gaps in a single point: {max_gaps}")
print(f"Min gaps in a single point: {min_gaps}")

📊 Final Dataset Gap Stats:
Average gaps: 1.59 per point
Max gaps in a single point: 5
Min gaps in a single point: 0


## export as csv file

In [78]:
# 1. Prepare Column Names
bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12', 'SCL', 'crop', 'mask']
all_col_names = []

# Create names like B2_0, B2_1, B2_2... for all 36 steps
for t in range(36):
    for b in bands:
        all_col_names.append(f"{b}_{t}")

# 2. Reshape the 3D array to 2D
# From (10000, 36, 13) -> (10000, 468)
flattened_data = balanced_dataset.reshape(balanced_dataset.shape[0], -1)

# 3. Create DataFrame and Save
df_final = pd.DataFrame(flattened_data, columns=all_col_names)

# Add a simple 'label' column at the end for easy filtering (optional but helpful)
# Since the crop ID is the same for all time steps, we just grab it from the first step
df_final['target_label'] = balanced_dataset[:, 0, 11]

df_final.to_csv("california.csv", index=False)

print(f"✅ Successfully saved {len(df_final)} samples to balanced_crop_dataset.csv")
print(f"Total columns: {len(df_final.columns)}")

✅ Successfully saved 10000 samples to balanced_crop_dataset.csv
Total columns: 469


### import csv script

In [ ]:
data = pd.read_csv("arkansas.csv")

# extract target
y = data["target_label"]
print (np.unique(y,return_counts=True))

# Remove the last column (target_label) and reshape to (10 000 , 36, 13)
features_only = data.values[:, :-1] 
original_shape = features_only.reshape(-1, 36, 13)

# Extract Spectral Bands (First 10 indices: B2 through B12)
X = original_shape[:, :, :10]
print(X[:5])

# 3. Extract Mask (The 13th index, which is index 12)
input2 = original_shape[:, :, 12]
print(input2[:5])


(array([0., 1., 2., 3., 5.]), array([ 616, 1522,  762, 2423, 4677]))
[[[ 2458.    1658.     634.   ...  1925.    2050.    2071.  ]
  [ 2927.    2207.     722.   ...  1806.    1994.    2031.  ]
  [ 2390.5   2288.    3144.5  ...  4026.    3986.    4054.5 ]
  ...
  [ 2636.    1726.     594.   ...  2582.    2864.    2902.  ]
  [ 2537.    1662.     570.   ...  2469.5   2692.    2736.5 ]
  [ 2438.    1598.     546.   ...  2357.    2520.    2571.  ]]

 [[ 2084.    1273.     521.   ...  2060.    2272.    2194.  ]
  [ 2889.    2133.     636.   ...  2311.    2576.    2479.  ]
  [ 3213.5   2716.    5791.   ...  6948.    6971.    6859.5 ]
  ...
  [ 2239.    1548.     744.   ...  1922.    2038.    2124.  ]
  [ 2016.    1406.75   763.   ...  1701.5   1842.5   1861.75]
  [ 1793.    1265.5    782.   ...  1481.    1647.    1599.5 ]]

 [[ 1956.    1148.     470.   ...  1191.    1426.    1577.  ]
  [ 2627.    1675.     330.   ...  1306.    1614.    1680.  ]
  [ 2951.    3096.5   5540.   ...  5843.    621